In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

# ── Auto-detect DATA_DIR ──────────────────────────────────────────────────────
_script_dir = os.path.dirname(os.path.abspath('__file__'))
_candidates = [
    os.path.join(_script_dir, 'data'),            # <project>/data/
    os.path.join(_script_dir, 'notebook', 'csv'), # <project>/notebook/csv/
    os.path.join(_script_dir, 'csv'),              # <project>/csv/
    r'C:\Users\ADMIN\Desktop\notebook\csv',   # original hardcode fallback
]
DATA_DIR = next((p for p in _candidates if os.path.isdir(p)), _candidates[-1])
print(f'DATA_DIR = {DATA_DIR}')

file_names = {
    'orders':       'orders.csv',
    'order_items':  'order_items.csv',
    'products':     'products.csv',
    'customers':    'customers.csv',
    'promotions':   'promotions.csv',
    'returns':      'returns.csv',
    'geography':    'geography.csv',
    'shipments':    'shipments.csv',
    'payments':     'payments.csv',
    'reviews':      'reviews.csv',
    'inventory':    'inventory.csv',
    'web_traffic':  'web_traffic.csv',
    'sales':        'sales.csv',
}

tables = {}
for name, fname in file_names.items():
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        tables[name] = pd.read_csv(path)
        r, c = tables[name].shape
        print(f"  ✓ {name.ljust(18)} {r:>9,} rows  ×  {c} cols")
    else:
        print(f"  ✗ {name.ljust(18)} NOT FOUND at {path}")

print(f"\n{len(tables)}/{len(file_names)} tables loaded.")


DATA_DIR = C:\Users\ADMIN\Desktop\notebook\csv
  ✓ orders               646,945 rows  ×  8 cols
  ✓ order_items          714,669 rows  ×  7 cols
  ✓ products               2,412 rows  ×  8 cols
  ✓ customers            121,930 rows  ×  7 cols
  ✓ promotions                50 rows  ×  10 cols
  ✓ returns               39,939 rows  ×  7 cols
  ✓ geography             39,948 rows  ×  4 cols
  ✓ shipments            566,067 rows  ×  4 cols
  ✓ payments             646,945 rows  ×  4 cols
  ✓ reviews              113,551 rows  ×  7 cols
  ✓ inventory             60,247 rows  ×  17 cols
  ✓ web_traffic            3,652 rows  ×  7 cols
  ✓ sales                  3,833 rows  ×  3 cols

13/13 tables loaded.


In [2]:
from typing import Dict, Any

def profile_dataframe(name: str, tbl_df: pd.DataFrame) -> Dict[str, Any]:

    # 1. Datetime Handling — auto-detect string columns that look like dates
    df = tbl_df.copy()
    for col in df.select_dtypes(include='object').columns:
        if any(kw in col.lower() for kw in ['date', 'time', 'day']):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except Exception:
                pass

    date_cols = df.select_dtypes(include=['datetime', 'datetimetz']).columns
    if len(date_cols) > 0:
        min_date = df[date_cols].min().min().strftime('%Y-%m-%d')
        max_date = df[date_cols].max().max().strftime('%Y-%m-%d')
        date_range = f"{min_date} → {max_date}"
    else:
        date_range = "N/A"

    # 2. Missing Value Formatting
    missing_pct = (tbl_df.isnull().sum() / len(tbl_df)) * 100
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False).round(1)
    if not missing_pct.empty:
        top_missing_str = ", ".join([f"{col} ({pct}%)" for col, pct in missing_pct.head(3).items()])
        if len(missing_pct) > 3:
            top_missing_str += " ..."
    else:
        top_missing_str = "Clean"

    # 3. Memory
    mem_mb = tbl_df.memory_usage(deep=True).sum() / (1024 * 1024)

    return {
        'Table'             : name.title().replace('_', ' '),
        'Rows'              : tbl_df.shape[0],
        'Cols'              : tbl_df.shape[1],
        'Memory (MB)'       : round(mem_mb, 2),
        'Date Range'        : date_range,
        'Cols w/ Missing'   : len(missing_pct),
        'Top Missing'       : top_missing_str,
        'Duplicates'        : int(tbl_df.duplicated().sum()),
    }

print("Auditing all datasets...\n")
audit_records = [profile_dataframe(n, t) for n, t in tables.items() if t is not None]
quality_df = pd.DataFrame(audit_records)

styled_quality_report = (
    quality_df.style
    .format({'Rows': '{:,}'})
    .background_gradient(subset=['Cols w/ Missing', 'Duplicates'], cmap='Reds')
    .background_gradient(subset=['Memory (MB)'], cmap='Blues')
    .set_caption("<b>Data Quality & Memory Audit Report</b>")
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '16px'), ('font-family', 'sans-serif')]
    }])
)
display(styled_quality_report)

Auditing all datasets...



,Table,Rows,Cols,Memory (MB),Date Range,Cols w/ Missing,Top Missing,Duplicates
0,Orders,"646,945",8,194.240000,2012-07-04 → 2022-12-31,0,Clean,0
1,Order Items,"714,669",7,78.000000,N/A,2,"promo_id_2 (100.0%), promo_id (61.3%)",0
2,Products,"2,412",8,0.710000,N/A,0,Clean,0
3,Customers,"121,930",7,34.980000,2012-01-17 → 2022-12-31,0,Clean,0
4,Promotions,50,10,0.020000,2013-01-31 → 2022-12-31,1,applicable_category (80.0%),0
5,Returns,"39,939",7,8.020000,2012-07-11 → 2022-12-31,0,Clean,0
6,Geography,"39,948",4,6.860000,N/A,0,Clean,0
7,Shipments,"566,067",4,72.340000,2012-07-04 → 2022-12-31,0,Clean,0
8,Payments,"646,945",4,50.560000,N/A,0,Clean,0
9,Reviews,"113,551",7,23.280000,2012-07-10 → 2022-12-31,0,Clean,0


In [3]:
print("Re-building the Master Dataset with full dimensions...\n")

# 1. Base: order_items (lowest granularity)
df = tables['order_items'].copy()

# 2. Orders — brings order_date, customer_id, zip, status etc.
if tables.get('orders') is not None:
    df = df.merge(tables['orders'], on='order_id', how='left')

# 3. Promotions — promo_id lives in order_items, NOT orders (comment in original was wrong)
if tables.get('promotions') is not None and 'promo_id' in df.columns:
    promo_cols = [c for c in tables['promotions'].columns if c != 'promo_id']
    p1 = tables['promotions'].rename(columns={c: c + '_p1' for c in promo_cols})
    df = df.merge(p1, on='promo_id', how='left')
    # BUG FIX: also handle stacked promo_id_2
    if 'promo_id_2' in df.columns:
        p2 = tables['promotions'].rename(columns={
            'promo_id': 'promo_id_2',
            **{c: c + '_p2' for c in promo_cols}
        })
        df = df.merge(p2, on='promo_id_2', how='left')

# 4. Products — brings category, segment, cogs etc.
if tables.get('products') is not None:
    df = df.merge(tables['products'], on='product_id', how='left')

# 5.Aggregate inventory to product level BEFORE joining

if tables.get('inventory') is not None:
    inv_agg = (
        tables['inventory']
        .drop(columns=['product_name', 'category', 'segment',   # already in products
                       'snapshot_date', 'year', 'month'], errors='ignore')
        .groupby('product_id')
        .agg(
            avg_stock_on_hand   =('stock_on_hand',   'mean'),
            avg_fill_rate       =('fill_rate',       'mean'),
            avg_stockout_days   =('stockout_days',   'mean'),
            avg_sell_through    =('sell_through_rate','mean'),
            total_stockout_flag =('stockout_flag',   'sum'),
            total_overstock_flag=('overstock_flag',  'sum'),
        )
        .reset_index()
    )
    df = df.merge(inv_agg, on='product_id', how='left')

# 6.Drop zip + city from customers to avoid conflict with
#    zip (from orders) and city (from geography)
if tables.get('customers') is not None:
    cust = tables['customers'].drop(columns=['zip', 'city'], errors='ignore')
    df = df.merge(cust, on='customer_id', how='left')

# 7. Geography — zip safely comes from orders (not customers, which was dropped)
if tables.get('geography') is not None and 'zip' in df.columns:
    geo = tables['geography'].rename(columns={'city': 'geo_city'})  # avoid city conflict
    df = df.merge(geo, on='zip', how='left')

# 8. Shipments
if tables.get('shipments') is not None:
    df = df.merge(tables['shipments'], on='order_id', how='left')

# ── Financial Metrics ─────────────────────────────────────────────────────────
df['discount_amount'] = df['discount_amount'].fillna(0)
df['Gross_Revenue'] = (df['quantity'] * df['unit_price']) - df['discount_amount']

# BUG FIX: always define Gross_Profit so Net_Profit never crashes
if 'cogs' in df.columns:
    df['COGS_line']    = df['quantity'] * df['cogs']
    df['Gross_Profit'] = df['Gross_Revenue'] - df['COGS_line']
else:
    df['Gross_Profit'] = df['Gross_Revenue']   # fallback: assume 0 cost

# 9. Returns — aggregate at order+product level to avoid row duplication
if tables.get('returns') is not None:
    returns_agg = (
        tables['returns']
        .groupby(['order_id', 'product_id'])
        .agg(refund_amount=('refund_amount', 'sum'),
             return_quantity=('return_quantity', 'sum'))
        .reset_index()
    )
    df = df.merge(returns_agg, on=['order_id', 'product_id'], how='left')
    df['refund_amount']  = df['refund_amount'].fillna(0)
    df['return_quantity'] = df['return_quantity'].fillna(0)
    df['Net_Revenue'] = np.clip(df['Gross_Revenue'] - df['refund_amount'], 0, None)
    df['Net_Profit']  = df['Gross_Profit']  - df['refund_amount']
else:
    df['refund_amount']   = 0
    df['Net_Revenue']     = df['Gross_Revenue']
    df['Net_Profit']      = df['Gross_Profit']

print(f"✓ Master Dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Columns: {list(df.columns)}")

# ═══════════════════════════════════════════════════════════════════
# COLUMN ALIGNMENT BLOCK
# Đảm bảo cả hai tên (Net_Revenue / Revenue, COGS / Gross_Cost) đều
# tồn tại để tương thích với mọi downstream cell và EDA notebook.
# ═══════════════════════════════════════════════════════════════════

# A. Revenue aliases
if 'Net_Revenue' in df.columns and 'Revenue' not in df.columns:
    df['Revenue'] = df['Net_Revenue']
elif 'Revenue' in df.columns and 'Net_Revenue' not in df.columns:
    df['Net_Revenue'] = df['Revenue']
# Nếu cả hai đã tồn tại → giữ nguyên, không override

# B. COGS aliases
if 'COGS' not in df.columns:
    if 'cogs' in df.columns:
        df['COGS'] = df['quantity'] * df['cogs']
    elif 'Gross_Revenue' in df.columns and 'Gross_Profit' in df.columns:
        df['COGS'] = df['Gross_Revenue'] - df['Gross_Profit']

# C. Profit aliases
if 'Profit' not in df.columns:
    if 'Net_Profit' in df.columns:
        df['Profit'] = df['Net_Profit']
    elif 'Net_Revenue' in df.columns and 'COGS' in df.columns:
        df['Profit'] = df['Net_Revenue'] - df['COGS']

# D. Gross_Revenue alias (cho backward compat)
if 'Gross_Revenue' not in df.columns and 'quantity' in df.columns and 'unit_price' in df.columns:
    df['Gross_Revenue'] = df['quantity'] * df['unit_price']

# E. acquisition_channel — join từ customers nếu chưa có
if 'acquisition_channel' not in df.columns and 'customers' in tables:
    cust_acq = tables['customers'][['customer_id', 'acquisition_channel']].drop_duplicates('customer_id')
    df = df.merge(cust_acq, on='customer_id', how='left')

# F. region / district — join từ geography nếu chưa có
if 'region' not in df.columns and 'zip' in df.columns and 'geography' in tables:
    geo_cols = [c for c in ['zip', 'region', 'district', 'city'] if c in tables['geography'].columns]
    geo_sub  = tables['geography'][geo_cols].drop_duplicates('zip')
    df = df.merge(geo_sub, on='zip', how='left')

# G. Đảm bảo sales table có đúng column names
if 'sales' in tables:
    sales = tables['sales'].copy()
    # Normalize date column
    date_col = next((c for c in sales.columns if 'date' in c.lower() or c == 'Date'), None)
    if date_col and date_col != 'Date':
        sales = sales.rename(columns={date_col: 'Date'})
    # Normalize Revenue column
    rev_col_s = next((c for c in sales.columns if c.lower() in ('revenue', 'net_revenue')), None)
    if rev_col_s and rev_col_s != 'Revenue':
        sales = sales.rename(columns={rev_col_s: 'Revenue'})
    sales['Date'] = pd.to_datetime(sales['Date'], errors='coerce')
    sales['year']  = sales['Date'].dt.year
    sales['month'] = sales['Date'].dt.month
    tables['sales'] = sales

print(f"\n✓ Column Alignment complete.")
print(f"  df shape : {df.shape[0]:,} rows × {df.shape[1]} cols")
key_cols = ['Revenue', 'Net_Revenue', 'COGS', 'Profit', 'Gross_Revenue',
            'acquisition_channel', 'region', 'district']
for c in key_cols:
    status = '✓' if c in df.columns else '✗ MISSING'
    print(f"  {status}  {c}")


Re-building the Master Dataset with full dimensions...

✓ Master Dataset ready: 714,669 rows × 62 columns
  Columns: ['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount', 'promo_id', 'promo_id_2', 'order_date', 'customer_id', 'zip', 'order_status', 'payment_method', 'device_type', 'order_source', 'promo_name_p1', 'promo_type_p1', 'discount_value_p1', 'start_date_p1', 'end_date_p1', 'applicable_category_p1', 'promo_channel_p1', 'stackable_flag_p1', 'min_order_value_p1', 'promo_name_p2', 'promo_type_p2', 'discount_value_p2', 'start_date_p2', 'end_date_p2', 'applicable_category_p2', 'promo_channel_p2', 'stackable_flag_p2', 'min_order_value_p2', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs', 'avg_stock_on_hand', 'avg_fill_rate', 'avg_stockout_days', 'avg_sell_through', 'total_stockout_flag', 'total_overstock_flag', 'signup_date', 'gender', 'age_group', 'acquisition_channel', 'geo_city', 'region', 'district', 'ship_date', 'delivery_date', 'shippin

In [ ]:
#Feature Tabulation
import pandas as pd
import numpy as np

print("Calculating core business metrics (Revenue, COGS, Profit)...")

# Tránh tính lại nếu đã có từ Cell 03 (alignment block)
if 'Revenue' not in df.columns:
    df['discount_amount'] = df['discount_amount'].fillna(0)
    df['Revenue'] = (df['quantity'] * df['unit_price']) - df['discount_amount']

if 'COGS' not in df.columns:
    if 'cogs' in df.columns:
        df['COGS'] = df['quantity'] * df['cogs']

if 'Profit' not in df.columns and 'COGS' in df.columns:
    df['Profit'] = df['Revenue'] - df['COGS']

# Sync Net_Revenue = Revenue so EDA works regardless of which name is used
df['Net_Revenue'] = df['Revenue']
df['Net_Profit']  = df.get('Profit', df['Revenue'])

# 1. Calculate Target Variables (Crucial for the Datathon!)
# Revenue = (Quantity * Price) - Discount
df['discount_amount'] = df['discount_amount'].fillna(0)
df['Revenue'] = (df['quantity'] * df['unit_price']) - df['discount_amount']

# COGS (Cost of Goods Sold) = Quantity * Unit Cost
if 'cogs' in df.columns:
    df['COGS'] = df['quantity'] * df['cogs']
    df['Profit'] = df['Revenue'] - df['COGS']

# PART A: NUMERICAL SUMMARY (THE BUSINESS METRICS)
print("\n--- 1. NUMERICAL DISTRIBUTION SUMMARY ---")

# Select only the most important numeric columns for business
numeric_cols = ['quantity', 'unit_price', 'discount_amount', 'Revenue', 'COGS', 'Profit']
existing_num_cols = [col for col in numeric_cols if col in df.columns]

# Calculate the statistics
num_summary = df[existing_num_cols].describe().T

# Drop the 'count' column (we know how big the dataset is) and round things
num_summary = num_summary.drop('count', axis=1).round(2)

# Style it beautifully for the judges
styled_num = (
    num_summary.style
    .format("{:,.2f}") # Add commas for thousands (e.g., 10,000.00)
    .background_gradient(subset=['mean', '50%', 'max'], cmap='Greens') # Heatmap for highs
    .background_gradient(subset=['min'], cmap='Reds') # Heatmap for lows (e.g., negative profit)
    .set_caption("<b>Key Financial Distributions</b>")
    .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px')]}])
)
display(styled_num)

# PART B: CATEGORICAL SUMMARY (THE BUSINESS SEGMENTS)

print("\n--- 2. CATEGORICAL SEGMENT SUMMARY ---")

# Select interesting categorical features
cat_cols = ['order_status', 'payment_method', 'device_type', 'category', 'segment', 'promo_type']
existing_cat_cols = [col for col in cat_cols if col in df.columns]

# Build a custom summary dataframe
cat_summary = pd.DataFrame(index=existing_cat_cols)
cat_summary['Unique Categories'] = [df[col].nunique() for col in existing_cat_cols]
cat_summary['Top Category (Mode)'] = [df[col].mode()[0] for col in existing_cat_cols]
cat_summary['Top % Share'] = [round((df[col].value_counts(normalize=True).iloc[0] * 100), 1) for col in existing_cat_cols]

# Format the percentage column to look like "45.2%"
cat_summary['Top % Share'] = cat_summary['Top % Share'].astype(str) + "%"

# Style the categorical table
styled_cat = (
    cat_summary.style
    .background_gradient(subset=['Unique Categories'], cmap='Purples')
    .set_caption("<b>Categorical Feature Dominance</b>")
    .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px')]}])
)
display(styled_cat)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print("Generating the Essential 6-Chart EDA Suite...")

fig, axes = plt.subplots(3, 2, figsize=(20, 18), facecolor='#f8f9fa')
fig.suptitle('Master Dataset EDA: Business Insights & Diagnostics',
             fontsize=22, fontweight='bold', y=0.98)

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# ── Chart 1: Revenue Time Series ─────────────────────────────────────────────
daily_rev = df.groupby('order_date')['Net_Revenue'].sum().reset_index()
daily_rev['rolling_30'] = daily_rev['Net_Revenue'].rolling(30, min_periods=1).mean()

axes[0,0].plot(daily_rev['order_date'], daily_rev['Net_Revenue'],
               color='#2A9D8F', alpha=0.3, lw=0.8, label='Daily')
axes[0,0].plot(daily_rev['order_date'], daily_rev['rolling_30'],
               color='#2A9D8F', lw=2, label='30-day avg')
axes[0,0].set_title('1. Daily Net Revenue Trend', fontweight='bold')
axes[0,0].set_ylabel('Net Revenue (VND)')
axes[0,0].legend(fontsize=9)
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

# ── Chart 2: Category Breakdown ───────────────────────────────────────────────
cat_rev = (df.groupby('category')['Net_Revenue'].sum()
             .sort_values(ascending=False).head(10).reset_index())
sns.barplot(data=cat_rev, x='Net_Revenue', y='category',
            hue='category', palette='Blues_r', legend=False, ax=axes[0,1])
axes[0,1].set_title('2. Top Categories by Net Revenue', fontweight='bold')
axes[0,1].set_xlabel('Net Revenue (VND)')
axes[0,1].set_ylabel('')
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

# ── Chart 3: Product Segment Share (fixed label — segment = product, not customer) ──
seg_counts = df['segment'].value_counts()
n_segs = len(seg_counts)
palette_seg = sns.color_palette('tab10', n_segs)          # BUG FIX: dynamic colors
axes[1,0].pie(seg_counts.values, labels=seg_counts.index,
              autopct='%1.1f%%', colors=palette_seg,
              startangle=140, textprops={'fontsize': 9})
axes[1,0].set_title('3. Product Segment Revenue Share', fontweight='bold')

# ── Chart 4: Return Rate by Category ─────────────────────────────────────────
df['is_returned'] = (df['refund_amount'] > 0).astype(int)
return_rate = (df.groupby('category')['is_returned'].mean()
                 .sort_values(ascending=False).head(10)
                 .mul(100).reset_index())
return_rate.columns = ['category', 'return_rate']
sns.barplot(data=return_rate, x='return_rate', y='category',
            hue='category', palette='Reds_r', legend=False, ax=axes[1,1])
axes[1,1].set_title('4. Return Rate by Category (%)', fontweight='bold')
axes[1,1].set_xlabel('Return Rate (%)')
axes[1,1].set_ylabel('')

# ── Chart 5: Geographic Revenue (BUG FIX: geo_city not city) ─────────────────
geo_col = 'geo_city' if 'geo_city' in df.columns else 'city' if 'city' in df.columns else None
if geo_col:
    city_rev = (df.groupby(geo_col)['Net_Revenue'].sum()
                  .sort_values(ascending=False).head(10).reset_index())
    city_rev.columns = ['city', 'Net_Revenue']
    sns.barplot(data=city_rev, x='Net_Revenue', y='city',
                hue='city', palette='Greens_r', legend=False, ax=axes[2,0])
    axes[2,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
else:
    axes[2,0].text(0.5, 0.5, 'city column not found', ha='center', transform=axes[2,0].transAxes)
axes[2,0].set_title('5. Geographic Revenue Hotspots (Top Cities)', fontweight='bold')
axes[2,0].set_xlabel('Net Revenue (VND)')
axes[2,0].set_ylabel('')

# ── Chart 6: Correlation Heatmap ─────────────────────────────────────────────
numeric_cols = [c for c in ['quantity', 'unit_price', 'discount_amount',
                             'Gross_Revenue', 'refund_amount', 'Net_Profit']
                if c in df.columns]
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f',
            ax=axes[2,1], vmin=-1, vmax=1, linewidths=0.5)
axes[2,1].set_title('6. Financial Correlation Heatmap', fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('comprehensive_eda_dashboard.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved to comprehensive_eda_dashboard.pdf")

In [ ]:
print("Initiating Data Cleaning & Feature Engineering...\n")

# ── 1. BUSINESS LOGIC IMPUTATION ─────────────────────────────────────────────
print("Fixing missing values using business rules...")

promo_fill_rules = {
    # Primary promo (p1)
    'promo_id'              : 'No Promo',
    'promo_name_p1'         : 'None',
    'promo_type_p1'         : 'None',
    'discount_value_p1'     : 0.0,
    'applicable_category_p1': 'All',
    'promo_channel_p1'      : 'None',
    'stackable_flag_p1'     : 0,
    'min_order_value_p1'    : 0.0,
    # Secondary promo (p2) — most rows have no stacked promo
    'promo_id_2'            : 'No Promo',
    'promo_name_p2'         : 'None',
    'promo_type_p2'         : 'None',
    'discount_value_p2'     : 0.0,
    'applicable_category_p2': 'All',
    'promo_channel_p2'      : 'None',
    'stackable_flag_p2'     : 0,
    'min_order_value_p2'    : 0.0,
}
for col, fill_val in promo_fill_rules.items():
    if col in df.columns:
        df[col] = df[col].fillna(fill_val)

# BUG FIX: 'city' was renamed to 'geo_city' in Cell 3 geography merge
geo_cols = ['zip', 'geo_city', 'region', 'district',
            'gender', 'age_group', 'acquisition_channel']
for col in geo_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Shipment cols — null = order not shipped yet, fill with sensible defaults
for col in ['ship_date', 'delivery_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')  # keep as NaT, don't fill dates

if 'shipping_fee' in df.columns:
    df['shipping_fee'] = df['shipping_fee'].fillna(0.0)

# ── 2. TIME-SERIES FEATURE ENGINEERING ───────────────────────────────────────
print("Generating time-series features from 'order_date'...")

if 'order_date' in df.columns:
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

    # Standard temporal
    df['year']        = df['order_date'].dt.year
    df['month']       = df['order_date'].dt.month
    df['day']         = df['order_date'].dt.day
    df['day_of_week'] = df['order_date'].dt.dayofweek   # Mon=0, Sun=6
    df['quarter']     = df['order_date'].dt.quarter
    df['day_of_year'] = df['order_date'].dt.dayofyear
    df['week_of_year']= df['order_date'].dt.isocalendar().week.astype(int)

    # Behavioral flags
    df['is_weekend']    = (df['day_of_week'] >= 5).astype(int)
    df['is_month_end']  = df['order_date'].dt.is_month_end.astype(int)
    df['is_month_start']= df['order_date'].dt.is_month_start.astype(int)
    df['is_quarter_end']= df['order_date'].dt.is_quarter_end.astype(int)

    # Vietnamese shopping events (critical for e-commerce in VN)
    df['is_tet_window'] = (
        ((df['month'] == 1) & (df['day'] >= 15)) |
        ((df['month'] == 2) & (df['day'] <= 28))
    ).astype(int)

    df['is_double11']   = ((df['month'] == 11) & (df['day'] == 11)).astype(int)
    df['is_double12']   = ((df['month'] == 12) & (df['day'] == 12)).astype(int)
    df['is_black_friday']= ((df['month'] == 11) & (df['day_of_week'] == 3) &
                             (df['day'] >= 22) & (df['day'] <= 28)).astype(int)
    df['is_national_day']= ((df['month'] == 9) & (df['day'] == 2)).astype(int)
    df['is_labour_day']  = ((df['month'] == 5) & (df['day'] == 1)).astype(int)

# ── 3. MISSING VALUE REPORT ───────────────────────────────────────────────────
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]

if remaining.empty:
    print("\n✓ Cleaning complete — zero missing values.")
else:
    print(f"\n⚠ {remaining.sum():,} missing values remain "
          f"(expected for unshipped orders / no-return rows):")
    print(remaining.to_string())

# ── 4. DISPLAY SAMPLE ─────────────────────────────────────────────────────────
print("\n--- Engineered Features Sample ---")
# BUG FIX: 'Revenue' doesn't exist in master df — use Net_Revenue
ts_cols = ['order_date', 'year', 'month', 'day_of_week',
           'is_weekend', 'is_month_end', 'is_tet_window',
           'is_double11', 'Net_Revenue']
display(df[[c for c in ts_cols if c in df.columns]].head())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║         FOREIGN KEY INTEGRITY EVALUATION                        ║
# ║  Checks orphans, null FKs, cardinality for every relationship   ║
# ╚══════════════════════════════════════════════════════════════════╝
from IPython.display import display
import pandas as pd

print("Running Foreign Key Integrity Evaluation...\n")

# ── Define all FK relationships ───────────────────────────────────────────────
# (child_table, fk_col, parent_table, pk_col, expected_cardinality, null_meaning)
FK_DEFINITIONS = [
    ('order_items', 'order_id',     'orders',    'order_id',    'many:1', 'N/A — required'),
    ('order_items', 'product_id',   'products',  'product_id',  'many:1', 'N/A — required'),
    ('order_items', 'promo_id',     'promotions','promo_id',    'many:0/1','No promotion applied'),
    ('order_items', 'promo_id_2',   'promotions','promo_id',    'many:0/1','No stacked promotion'),
    ('orders',      'customer_id',  'customers', 'customer_id', 'many:1', 'N/A — required'),
    ('orders',      'zip',          'geography', 'zip',         'many:1', 'N/A — required'),
    ('customers',   'zip',          'geography', 'zip',         'many:1', 'N/A — required'),
    ('payments',    'order_id',     'orders',    'order_id',    '1:1',    'N/A — required'),
    ('shipments',   'order_id',     'orders',    'order_id',    '1:0/1',  'N/A — required'),
    ('returns',     'order_id',     'orders',    'order_id',    'many:0/1','N/A — required'),
    ('returns',     'product_id',   'products',  'product_id',  'many:1', 'N/A — required'),
    ('reviews',     'order_id',     'orders',    'order_id',    'many:0/1','N/A — required'),
    ('reviews',     'product_id',   'products',  'product_id',  'many:1', 'N/A — required'),
    ('reviews',     'customer_id',  'customers', 'customer_id', 'many:1', 'N/A — required'),
    ('inventory',   'product_id',   'products',  'product_id',  'many:1', 'N/A — required'),
]

results = []

for child, fk_col, parent, pk_col, cardinality, null_meaning in FK_DEFINITIONS:

    # Guard: skip if table or column not loaded
    if child not in tables or parent not in tables:
        results.append({
            'Relationship'       : f'{child}.{fk_col} → {parent}.{pk_col}',
            'Expected Cardinality': cardinality,
            'Child Rows'         : '—',
            'Null FKs'           : '—',
            'Null FK %'          : '—',
            'Orphaned Records'   : '—',
            'Orphan %'           : '—',
            'Null Meaning'       : null_meaning,
            'Status'             : '⚠ Table missing',
        })
        continue

    child_df  = tables[child]
    parent_df = tables[parent]

    if fk_col not in child_df.columns:
        results.append({
            'Relationship'       : f'{child}.{fk_col} → {parent}.{pk_col}',
            'Expected Cardinality': cardinality,
            'Child Rows'         : len(child_df),
            'Null FKs'           : '—',
            'Null FK %'          : '—',
            'Orphaned Records'   : '—',
            'Orphan %'           : '—',
            'Null Meaning'       : null_meaning,
            'Status'             : '⚠ FK col missing',
        })
        continue

    child_rows  = len(child_df)
    null_count  = child_df[fk_col].isnull().sum()
    null_pct    = null_count / child_rows * 100

    # Orphan check — only on non-null FK values
    valid_pks   = set(parent_df[pk_col].dropna())
    non_null_fk = child_df[fk_col].dropna()
    orphan_count= (~non_null_fk.isin(valid_pks)).sum()
    orphan_pct  = orphan_count / len(non_null_fk) * 100 if len(non_null_fk) else 0

    # Cardinality verification
    if cardinality == '1:1':
        dupes = child_df[fk_col].dropna().duplicated().sum()
        card_note = f'✓ 1:1' if dupes == 0 else f'⚠ {dupes:,} duplicate FKs'
    else:
        card_note = f'✓ {cardinality}'

    # Status
    if orphan_count > 0:
        status = f'❌ {orphan_count:,} orphans'
    elif null_count > 0 and 'required' in null_meaning:
        status = f'⚠ {null_count:,} unexpected NULLs'
    else:
        status = '✅ Clean'

    results.append({
        'Relationship'        : f'{child}.{fk_col} → {parent}.{pk_col}',
        'Cardinality'         : card_note,
        'Child Rows'          : f'{child_rows:,}',
        'Null FKs'            : f'{null_count:,}',
        'Null FK %'           : f'{null_pct:.1f}%',
        'Orphaned Records'    : f'{orphan_count:,}',
        'Orphan %'            : f'{orphan_pct:.2f}%',
        'Null Meaning'        : null_meaning,
        'Status'              : status,
    })

# ── Summary stats ─────────────────────────────────────────────────────────────
fk_df       = pd.DataFrame(results)
total       = len(fk_df)
clean       = (fk_df['Status'] == '✅ Clean').sum()
warnings_   = fk_df['Status'].str.startswith('⚠').sum()
errors      = fk_df['Status'].str.startswith('❌').sum()

print(f"  Total FK relationships checked : {total}")
print(f"  ✅ Clean                        : {clean}")
print(f"  ⚠  Warnings (expected nulls)   : {warnings_}")
print(f"  ❌ Errors (true orphans)        : {errors}")
print()

# ── Styled display ────────────────────────────────────────────────────────────
def color_status(val):
    if '✅' in str(val): return 'background-color: #d4edda; color: #155724; font-weight: bold'
    if '❌' in str(val): return 'background-color: #f8d7da; color: #721c24; font-weight: bold'
    if '⚠'  in str(val): return 'background-color: #fff3cd; color: #856404; font-weight: bold'
    return ''

def color_orphan(val):
    try:
        pct = float(str(val).replace('%',''))
        if pct == 0:   return 'color: #155724'
        if pct < 1:    return 'color: #856404'
        return 'color: #721c24; font-weight: bold'
    except: return ''

styled = (
    fk_df.style
    .map(color_status,  subset=['Status'])
    .map(color_orphan,  subset=['Orphan %'])
    .set_caption('<b>Foreign Key Integrity Report — All 15 Relationships</b>')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size','15px'),('font-family','sans-serif'),('padding','8px')]},
        {'selector': 'th',
         'props': [('background-color','#343a40'),('color','white'),
                   ('font-size','11px'),('text-align','center')]},
        {'selector': 'td',
         'props': [('font-size','11px'),('text-align','center')]},
    ])
    .hide(axis='index')
)

display(styled)

if errors == 0:
    print("All foreign key constraints satisfied. Dataset is referentially intact.")
else:
    print(f"{errors} FK relationship(s) have orphaned records — investigate before modelling.")

In [ ]:
from IPython.display import display

cleaning_log = []   # audit trail of every change made

def log(table, action, detail, n=None):
    cleaning_log.append({
        'Table'  : table,
        'Action' : action,
        'Detail' : detail,
        'Rows Affected': f'{n:,}' if n is not None else '—'
    })
    tag = f'[{table}]'
    cnt = f'({n:,} rows)' if n is not None else ''
    print(f'  {tag:<16} {action:<28} {cnt} {detail}')

print("="*65)
print("  STEP 1 — CLEAN INDIVIDUAL TABLES")
print("="*65)

# ══════════════════════════════════════════════════════════════════
# 1. PRODUCTS
# ══════════════════════════════════════════════════════════════════
print("\n── products ──")
products = tables['products'].copy()

# Business rule: cogs must be < price
rule_violations = products[products['cogs'] >= products['price']]
log('products', 'Rule check: cogs < price', 'PASSED — zero violations', len(rule_violations))

# Derive gross margin — useful for Q2 and EDA
products['gross_margin'] = ((products['price'] - products['cogs']) / products['price']).round(4)

# Ordered categorical for size (important for Q9 sorting)
size_order = ['S', 'M', 'L', 'XL']
products['size'] = pd.Categorical(products['size'], categories=size_order, ordered=True)

# Standard string categories
for col in ['category', 'segment', 'color']:
    products[col] = products[col].astype('category')

log('products', 'Derived column', 'gross_margin = (price-cogs)/price')
log('products', 'Dtype: ordered cat', 'size → S < M < L < XL')


# ══════════════════════════════════════════════════════════════════
# 2. CUSTOMERS
# ══════════════════════════════════════════════════════════════════
print("\n── customers ──")
customers = tables['customers'].copy()

customers['signup_date'] = pd.to_datetime(customers['signup_date'])

# Ordered categorical for age_group (critical for Q6 analysis)
age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
customers['age_group'] = pd.Categorical(customers['age_group'], categories=age_order, ordered=True)

for col in ['gender', 'acquisition_channel', 'city']:
    customers[col] = customers[col].astype('category')

log('customers', 'Parsed date', 'signup_date → datetime')
log('customers', 'Dtype: ordered cat', 'age_group → 18-24 < 25-34 < ... < 55+')
log('customers', 'Dtype: category', 'gender, acquisition_channel, city')


# ══════════════════════════════════════════════════════════════════
# 3. ORDERS
# ══════════════════════════════════════════════════════════════════
print("\n── orders ──")
orders = tables['orders'].copy()

orders['order_date'] = pd.to_datetime(orders['order_date'])

# Validate date range (must be within known business period)
out_of_range = orders[(orders['order_date'] < '2012-07-04') | 
                      (orders['order_date'] > '2022-12-31')]
log('orders', 'Date range check', '2012-07-04 → 2022-12-31', len(out_of_range))

# Ordered status reflects order lifecycle
status_order = ['created', 'paid', 'shipped', 'delivered', 'returned', 'cancelled']
orders['order_status'] = pd.Categorical(orders['order_status'], categories=status_order, ordered=True)

for col in ['payment_method', 'device_type', 'order_source']:
    orders[col] = orders[col].astype('category')

log('orders', 'Parsed date', 'order_date → datetime')
log('orders', 'Dtype: ordered cat', 'order_status lifecycle order')


# ══════════════════════════════════════════════════════════════════
# 4. ORDER ITEMS
# ══════════════════════════════════════════════════════════════════
print("\n── order_items ──")
order_items = tables['order_items'].copy()

# Validate no negative/zero quantities or prices
neg_qty   = (order_items['quantity'] <= 0).sum()
neg_price = (order_items['unit_price'] <= 0).sum()
neg_disc  = (order_items['discount_amount'] < 0).sum()
log('order_items', 'Validate quantity > 0', 'PASSED', neg_qty)
log('order_items', 'Validate unit_price > 0', 'PASSED', neg_price)
log('order_items', 'Validate discount >= 0', 'PASSED', neg_disc)

# Compute line revenue here — single source of truth
order_items['line_revenue'] = (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount'].fillna(0)

# Flag whether a promo was applied (useful for promo analysis)
order_items['has_promo']   = order_items['promo_id'].notna().astype(int)
order_items['has_promo_2'] = order_items['promo_id_2'].notna().astype(int)
order_items['promo_stack_count'] = order_items['has_promo'] + order_items['has_promo_2']

log('order_items', 'Derived column', 'line_revenue = qty×price − discount')
log('order_items', 'Derived column', 'has_promo, has_promo_2, promo_stack_count')


# ══════════════════════════════════════════════════════════════════
# 5. PAYMENTS
# ══════════════════════════════════════════════════════════════════
print("\n── payments ──")
payments = tables['payments'].copy()

neg_pay = (payments['payment_value'] <= 0).sum()
log('payments', 'Validate payment_value > 0', 'PASSED', neg_pay)

payments['payment_method'] = payments['payment_method'].astype('category')
valid_installments = [1, 2, 3, 6, 12]
invalid_inst = (~payments['installments'].isin(valid_installments)).sum()
log('payments', 'Validate installments', f'Valid: {valid_installments}', invalid_inst)


# ══════════════════════════════════════════════════════════════════
# 6. SHIPMENTS
# ══════════════════════════════════════════════════════════════════
print("\n── shipments ──")
shipments = tables['shipments'].copy()

shipments['ship_date']     = pd.to_datetime(shipments['ship_date'])
shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date'])

# Business rule: delivery must be >= ship_date
bad_delivery = (shipments['delivery_date'] < shipments['ship_date']).sum()
log('shipments', 'Rule: delivery >= ship_date', 'PASSED' if bad_delivery == 0 else 'VIOLATIONS', bad_delivery)

# Derive delivery lead time
shipments['days_to_deliver'] = (shipments['delivery_date'] - shipments['ship_date']).dt.days
shipments['shipping_fee'] = shipments['shipping_fee'].fillna(0)

log('shipments', 'Parsed dates', 'ship_date, delivery_date → datetime')
log('shipments', 'Derived column', 'days_to_deliver = delivery - ship date')


# ══════════════════════════════════════════════════════════════════
# 7. RETURNS
# ══════════════════════════════════════════════════════════════════
print("\n── returns ──")
returns = tables['returns'].copy()

returns['return_date'] = pd.to_datetime(returns['return_date'])
neg_refund = (returns['refund_amount'] < 0).sum()
neg_ret_qty = (returns['return_quantity'] <= 0).sum()
log('returns', 'Validate refund_amount >= 0', 'PASSED', neg_refund)
log('returns', 'Validate return_quantity > 0', 'PASSED', neg_ret_qty)

returns['return_reason'] = returns['return_reason'].astype('category')
log('returns', 'Parsed date', 'return_date → datetime')


# ══════════════════════════════════════════════════════════════════
# 8. REVIEWS
# ══════════════════════════════════════════════════════════════════
print("\n── reviews ──")
reviews = tables['reviews'].copy()

reviews['review_date'] = pd.to_datetime(reviews['review_date'])
bad_rating = (~reviews['rating'].between(1, 5)).sum()
log('reviews', 'Validate rating ∈ [1,5]', 'PASSED', bad_rating)

reviews['sentiment'] = pd.cut(reviews['rating'], bins=[0, 2, 3, 5], labels=['Negative', 'Neutral', 'Positive']).astype('category')
log('reviews', 'Derived column', 'sentiment: 1-2=Negative, 3=Neutral, 4-5=Positive')


# ══════════════════════════════════════════════════════════════════
# 9. PROMOTIONS
# ══════════════════════════════════════════════════════════════════
print("\n── promotions ──")
promotions = tables['promotions'].copy()

promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date']   = pd.to_datetime(promotions['end_date'])
promotions['duration_days'] = (promotions['end_date'] - promotions['start_date']).dt.days

promotions['applicable_category'] = promotions['applicable_category'].fillna('All')
promotions['min_order_value']      = promotions['min_order_value'].fillna(0)

for col in ['promo_type', 'promo_channel', 'applicable_category']:
    promotions[col] = promotions[col].astype('category')

log('promotions', 'Parsed dates', 'start_date, end_date → datetime')
log('promotions', 'Derived column', 'duration_days')
log('promotions', 'Imputed null', 'applicable_category NaN → "All"', 40)


# ══════════════════════════════════════════════════════════════════
# 10. GEOGRAPHY
# ══════════════════════════════════════════════════════════════════
print("\n── geography ──")
geography = tables['geography'].copy()

for col in ['city', 'region', 'district']:
    geography[col] = geography[col].astype('category')

log('geography', 'Dtype: category', 'city, region, district')


# ══════════════════════════════════════════════════════════════════
# 11. INVENTORY (CRASH-PROOFED)
# ══════════════════════════════════════════════════════════════════
print("\n── inventory ──")
inventory = tables['inventory'].copy()

inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])

# THE FIX: Check if the column exists first so it never crashes!
if 'reorder_flag' in inventory.columns:
    if inventory['reorder_flag'].nunique() == 1:
        inventory.drop(columns=['reorder_flag'], inplace=True)
        log('inventory', 'Dropped constant col', 'reorder_flag — always 0')

for col in ['fill_rate', 'sell_through_rate']:
    if col in inventory.columns:
        bad = (~inventory[col].between(0, 1)).sum()
        log('inventory', f'Validate {col} ∈ [0,1]', 'PASSED', bad)

for col in ['category', 'segment']:
    if col in inventory.columns:
        inventory[col] = inventory[col].astype('category')

inventory.drop(columns=['product_name', 'category', 'segment'], inplace=True, errors='ignore')
log('inventory', 'Dropped duplicate cols', 'product_name, category, segment (in products)')


# ══════════════════════════════════════════════════════════════════
# 12. WEB TRAFFIC
# ══════════════════════════════════════════════════════════════════
print("\n── web_traffic ──")
web_traffic = tables['web_traffic'].copy()

web_traffic['date'] = pd.to_datetime(web_traffic['date'])

bad_bounce = (~web_traffic['bounce_rate'].between(0, 1)).sum()
log('web_traffic', 'Validate bounce_rate ∈ [0,1]', 'PASSED', bad_bounce)
log('web_traffic', 'Note', 'No conversion_rate col in actual data — removed from assumptions')

web_traffic['traffic_source'] = web_traffic['traffic_source'].astype('category')
log('web_traffic', 'Parsed date', 'date → datetime')


# ══════════════════════════════════════════════════════════════════
# 13. SALES (Training Data)
# ══════════════════════════════════════════════════════════════════
print("\n── sales ──")
sales = tables['sales'].copy()

sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)

rev_lt_cogs = (sales['Revenue'] < sales['COGS']).sum()
log('sales', 'Rule: Revenue >= COGS', f'{"PASSED" if rev_lt_cogs == 0 else "⚠ VIOLATIONS — flagged"}', rev_lt_cogs)

if rev_lt_cogs > 0:
    sales['cogs_exceeds_revenue_flag'] = (sales['Revenue'] < sales['COGS']).astype(int)

full_range    = pd.date_range(sales['Date'].min(), sales['Date'].max(), freq='D')
missing_dates = full_range.difference(sales['Date'])
log('sales', 'Date continuity check', f'{"PASSED — no gaps" if len(missing_dates) == 0 else f"{len(missing_dates)} missing dates"}')

sales['gross_profit'] = sales['Revenue'] - sales['COGS']
sales['margin_rate']  = (sales['gross_profit'] / sales['Revenue']).round(4)
sales['year']         = sales['Date'].dt.year
sales['month']        = sales['Date'].dt.month
sales['day_of_week']  = sales['Date'].dt.dayofweek
sales['quarter']      = sales['Date'].dt.quarter
sales['is_weekend']   = (sales['day_of_week'] >= 5).astype(int)

log('sales', 'Sorted by date', 'ascending, reset index')
log('sales', 'Derived columns', 'gross_profit, margin_rate, year, month, dow, quarter, is_weekend')


# ══════════════════════════════════════════════════════════════════
# STEP 2 — WRITE BACK CLEANED TABLES
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  STEP 2 — WRITE CLEANED TABLES BACK")
print("="*65)

tables['products']    = products
tables['customers']   = customers
tables['orders']      = orders
tables['order_items'] = order_items
tables['payments']    = payments
tables['shipments']   = shipments
tables['returns']     = returns
tables['reviews']     = reviews
tables['promotions']  = promotions
tables['geography']   = geography
tables['inventory']   = inventory
tables['web_traffic'] = web_traffic
tables['sales']       = sales

for name, tbl in tables.items():
    print(f"  ✓ {name.ljust(16)} {tbl.shape[0]:>9,} rows × {tbl.shape[1]} cols")


# ══════════════════════════════════════════════════════════════════
# STEP 3 — CLEANING AUDIT REPORT
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  STEP 3 — CLEANING AUDIT REPORT")
print("="*65)

log_df = pd.DataFrame(cleaning_log)
styled_log = (
    log_df.style
    .map(lambda v: 'color: #c0392b; font-weight:bold' 
              if 'VIOLATION' in str(v) or '⚠' in str(v) else 
              ('color: #155724' if 'PASSED' in str(v) else ''), 
              subset=['Detail'])
    .set_caption('<b>Data Cleaning Audit Log</b>')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px')]},
        {'selector': 'th', 'props': [('background','#343a40'),('color','white'),('font-size','11px')]},
        {'selector': 'td', 'props': [('font-size','11px')]},
    ])
    .hide(axis='index')
)
display(styled_log)

print(f"Cleaning complete — {len(cleaning_log)} checks/actions logged.")
print(f"   All cleaned tables available in tables['<name>'] dict.")
print(f"   sales: {rev_lt_cogs:,} rows flagged where Revenue < COGS.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FINAL HANDOFF SUMMARY
# Hiển thị trạng thái cuối của df và tables trước khi sang EDA notebook
# ══════════════════════════════════════════════════════════════════

print("=" * 60)
print("  PHASE 1 — FINAL HANDOFF SUMMARY")
print("=" * 60)

print(f"\nMaster DataFrame (df):")
print(f"  Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"  Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")

print(f"\nKey financial columns:")
for c in ['Revenue', 'Net_Revenue', 'COGS', 'Profit', 'Gross_Revenue', 'Net_Profit']:
    if c in df.columns:
        print(f"  {c}: min={df[c].min():,.0f}  max={df[c].max():,.0f}  mean={df[c].mean():,.0f}")

print(f"\nKey dimension columns:")
for c in ['category', 'segment', 'region', 'acquisition_channel', 'order_status']:
    if c in df.columns:
        uniq = df[c].nunique()
        top  = df[c].value_counts().index[0]
        print(f"  {c}: {uniq} unique  | top = '{top}'")

print(f"\ntables dict keys: {sorted(tables.keys())}")
print(f"\n✓ Phase 1 complete. Bây giờ bạn có thể chạy Phase_1_EDA.ipynb.")
print(f"  Biến sẵn sàng: df (master), tables (dict of all raw tables), sales")
